# Name That LLM — Optimized character n-gram sweep

This notebook evaluates a lightweight authorship-attribution model under prompt-grouped cross-validation.

**Fixed features:** word TF-IDF + punctuation frequencies.
**Swept feature:** raw character TF-IDF n-gram range.
**Classifier:** class-balanced LinearSVC.

The word matrix, punctuation matrix, and grouped folds are reused across configurations to reduce redundant computation.


In [ ]:
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# ------------------------------
# Reproducible experiment config
# ------------------------------
DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
OUTPUT_PATH = Path('/kaggle/working/char_sweep_results.json')
N_SPLITS = 5
RANDOM_STATE = 42
WORD_NGRAM_RANGE = (1, 3)
CHAR_NGRAM_RANGES = [(1, 5), (2, 6), (3, 7), (2, 7)]
MIN_DF = 2
MAX_FEATURES = 120_000
SVC_C = 1.0
MAX_ITER = 1_500

assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found: {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
required = {'text', 'model', 'prompt_id'}
missing = required.difference(train.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'

texts = train['text'].fillna('').astype(str).to_numpy()
labels = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()

print(f'Train rows: {len(train):,}')
print(f'Classes: {len(np.unique(labels))}')
print(f'Prompt groups: {len(np.unique(groups))}')
print(f'Character ranges: {CHAR_NGRAM_RANGES}')


In [ ]:
# ------------------------------
# Feature engineering utilities
# ------------------------------
PUNCTUATION = tuple('.,;:!?()[]{}"\'/-')

def punctuation_matrix(items):
    """Return per-character punctuation frequencies, one row per response."""
    rows = []
    for text in items:
        denominator = max(len(text), 1)
        rows.append([text.count(char) / denominator for char in PUNCTUATION])
    return np.asarray(rows, dtype=np.float32)

def make_word_vectorizer():
    return TfidfVectorizer(
        analyzer='word',
        ngram_range=WORD_NGRAM_RANGE,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def make_char_vectorizer(ngram_range):
    return TfidfVectorizer(
        analyzer='char',
        ngram_range=ngram_range,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def fit_score(X_train, y_train, X_valid, y_valid):
    model = LinearSVC(
        C=SVC_C,
        class_weight='balanced',
        max_iter=MAX_ITER,
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_valid)
    return float(balanced_accuracy_score(y_valid, predictions))

punctuation = punctuation_matrix(texts)
folds = list(GroupKFold(n_splits=N_SPLITS).split(texts, labels, groups=groups))
print(f'Prepared {len(folds)} fixed prompt-grouped folds.')


In [ ]:
# ------------------------------
# Optimized grouped CV sweep
# ------------------------------
# Word TF-IDF and punctuation are fitted/transformed once per fold.
# Only the character vectorizer is refit for each candidate range.
results = {}
experiment_started = time.time()

for char_range in CHAR_NGRAM_RANGES:
    range_key = f'char_{char_range[0]}_{char_range[1]}'
    fold_scores = []
    fold_started = time.time()

    for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
        y_train, y_valid = labels[train_idx], labels[valid_idx]

        # Fit word features once for this fold and reuse for all ranges.
        word_vectorizer = make_word_vectorizer()
        X_word_train = word_vectorizer.fit_transform(texts[train_idx])
        X_word_valid = word_vectorizer.transform(texts[valid_idx])

        # Scale only the dense punctuation block using training statistics.
        punctuation_scaler = StandardScaler()
        X_punct_train = csr_matrix(punctuation_scaler.fit_transform(punctuation[train_idx]))
        X_punct_valid = csr_matrix(punctuation_scaler.transform(punctuation[valid_idx]))

        char_vectorizer = make_char_vectorizer(char_range)
        X_char_train = char_vectorizer.fit_transform(texts[train_idx])
        X_char_valid = char_vectorizer.transform(texts[valid_idx])

        X_train = hstack([X_word_train, X_char_train, X_punct_train], format='csr')
        X_valid = hstack([X_word_valid, X_char_valid, X_punct_valid], format='csr')

        score = fit_score(X_train, y_train, X_valid, y_valid)
        fold_scores.append(score)
        print(f'{range_key} | fold {fold_number}/{N_SPLITS} | score={score:.4f} | features={X_train.shape[1]:,}')

    results[range_key] = {
        'char_ngram_range': list(char_range),
        'mean_balanced_accuracy': float(np.mean(fold_scores)),
        'std_balanced_accuracy': float(np.std(fold_scores, ddof=1)),
        'fold_scores': fold_scores,
        'elapsed_seconds': round(time.time() - fold_started, 2),
    }
    print(f'{range_key} | mean={np.mean(fold_scores):.4f} | std={np.std(fold_scores, ddof=1):.4f}')

ranking = sorted(results.items(), key=lambda item: item[1]['mean_balanced_accuracy'], reverse=True)
summary = {
    'experiment': 'optimized character n-gram sweep with word TF-IDF and punctuation',
    'split_method': '5-fold GroupKFold by prompt_id',
    'config': {
        'word_ngram_range': list(WORD_NGRAM_RANGE),
        'char_ngram_ranges': [list(x) for x in CHAR_NGRAM_RANGES],
        'min_df': MIN_DF,
        'max_features_per_vectorizer': MAX_FEATURES,
        'linear_svc_C': SVC_C,
        'max_iter': MAX_ITER,
    },
    'results': results,
    'ranking': [{'configuration': key, **value} for key, value in ranking],
    'total_elapsed_seconds': round(time.time() - experiment_started, 2),
}
OUTPUT_PATH.write_text(json.dumps(summary, indent=2))
print('\nFinal ranking:')
for position, (key, value) in enumerate(ranking, start=1):
    print(f'{position}. {key}: {value["mean_balanced_accuracy"]:.4f} +/- {value["std_balanced_accuracy"]:.4f}')
print(f'Saved results to {OUTPUT_PATH}')


## Interpretation

Select the character range using the mean score **and** fold stability. The selected range will be kept fixed for the next LinearSVC `C` sweep. The final candidate must be rechecked on a separate grouped verification split before submission.
